# 5. Relationships, and what makes one believable

**The question: does this pattern mean anything?**

Lesson 3 subtracted the boring part, lesson 4 fitted what was left. This lesson is about the
step everybody skips — deciding whether the thing you found is real *before* writing it down.

Three moves, in order:

1. **The line you draw is a claim.** A fitted line says what *kind* of relationship you
   believe in, and the picture tells you whether that belief was earned.
2. **Walk into the trap.** Hunt for a difference between two groups that are not different.
   You will find one, about half the time, and the arithmetic says why.
3. **A grid for deciding what a finding is worth**, run on three claims that get three
   different verdicts.

Summary statistics agree far more often than pictures do — Anscombe's quartet made that point
in 1973 and you have seen it. The habit it buys is the one this notebook uses throughout: plot
it before you summarise it, and plot it again after.

The rest of the lesson takes one claim all the way through the grid
([05.4](05.4-how-people-type.ipynb)), meets the confounder that no method fixes
([05.2](05.2-correlation.ipynb)), and turns a notebook into a script
([05.3](05.3-notebook-to-script.ipynb)).

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import (
    Filter,
    FlagDates,
    GroupAgg,
    Pipeline,
    RegexFeature,
)
from goad_toolkit.visualizer import (
    BarbellPlot,
    BarPlot,
    CorrelationHeatmap,
    GroupedBarPlot,
    HighlightCategory,
    HistogramPlot,
    HorizontalLine,
    LinePlot,
    PlotSettings,
    RegPlot,
    ScatterPlot,
    VerticalLine,
)
from scipy import stats

from scripts.pipelines import build_irc_pipeline
from wa_analyzer.data import load_showcase

## 5.1 The line you draw is a claim

A scatter shows the relationship. A fitted line says what *kind* it is — and that is a claim
you are making, not a formatting choice. `RegPlot` takes the three that matter: `fit_reg`
for whether to draw one at all, `order` for a polynomial, `lowess` to let the data pick the
shape.

Fuel efficiency against weight, which is famously not a straight line. The points are grey on
purpose: the line is the claim, so the line gets the colour.

In [ ]:
mpg = load_showcase("mpg").dropna(subset=["weight", "mpg"])

shapes = PlotSettings(
    figsize=(13, 3.6),  # ty: ignore[invalid-argument-type]
    title="Three claims about the same scatter",
    subplot_titles=["order=1: a straight line", "order=2: a curve", "lowess: no shape assumed"],
    xlabel="weight (lbs)",
    ylabel="miles per gallon",
)
points = {"scatter_kws": {"color": "#bbbbbb", "s": 12}}

host = ScatterPlot(shapes)
fig, axes = host.create_figure(n_plots=3)
host.plot_on_axes(RegPlot(shapes), axes[0], data=mpg, x="weight", y="mpg", ci=None, **points)
host.plot_on_axes(RegPlot(shapes), axes[1], data=mpg, x="weight", y="mpg", order=2, ci=None, **points)
_ = host.plot_on_axes(RegPlot(shapes), axes[2], data=mpg, x="weight", y="mpg", lowess=True, **points)

The straight line is wrong in a specific, readable way: it over-predicts economy for the
heaviest cars and under-predicts it for the lightest, because it is averaging a curve. The
quadratic follows the bend. Lowess agrees with the quadratic without being told there was a
bend to find.

**Which to use.** `lowess` first, when you do not know the shape — it is a description. Then
a polynomial once you have decided what the shape *is* — that is a model, it has parameters,
and you can extrapolate from it and be wrong in an informative way.

The numbers behind the lines: `scipy.stats.linregress` for the straight one, and the same
fit on log-log axes, because "each extra pound costs proportionally less" is a claim about
*ratios* rather than differences.

In [ ]:
fit = stats.linregress(mpg.weight, mpg.mpg)
log_fit = stats.linregress(np.log(mpg.weight), np.log(mpg.mpg))
quad = np.polyfit(mpg.weight, mpg.mpg, 2)

print(f"straight line  slope {fit.slope:.5f} mpg per lb   r^2 {fit.rvalue**2:.3f}   p {fit.pvalue:.1e}")
print(f"log-log line   slope {log_fit.slope:.3f}             r^2 {log_fit.rvalue**2:.3f}")
print(f"quadratic      {quad[0]:.3e} x^2 + {quad[1]:.3f} x + {quad[2]:.1f}")

`r² = 0.69` and a p-value with a lot of zeros in it — both true, and neither tells you the
relationship is curved, which the picture said immediately. The log-log fit says mpg falls
with roughly the **1.2th power** of weight, and its r² beats the straight line on the raw
scale. That is a sentence about mechanism — doubling the weight costs more than half the
economy — rather than a slope in units nobody thinks in.

> **Your turn, briefly.** Fit `order=3` to the same data. Does it follow the points better?
> Does it *predict* better? Those are different questions, and lesson 6 is where the second
> one gets its own machinery.

## 5.2 The trap: a finding about nothing

**The insight this section sells:** search fifteen metrics for a difference between two
groups, and you will find a "significant" one about half the time *when there is nothing to
find*. Everything below is that one sentence, shown and then explained.

The IRC corpus, one row per `#ubuntu-uk` author with thirty or more messages, and fifteen
metrics per author: how long their messages are, how often they post at night, how many
questions and exclamation marks, how many links, and so on. The pattern-shaped features are
`RegexFeature` steps in a pipeline — lesson 1's move — while the one-liners (a length, a clock
flag) are assigned directly. Each author also carries a `cohort` label, A or B.

In [ ]:
msgs = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))

uk = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-uk'")
    .add(RegexFeature, name="words", column="message", pattern=r"\S+",
         feature="words", mode="count")
    .add(RegexFeature, name="caps", column="message", pattern=r"^[A-Z]",
         feature="starts_upper", mode="has")
    .add(RegexFeature, name="exclaims", column="message", pattern="!",
         feature="exclaims", mode="count")
    .apply(msgs)
)
uk["length"] = uk.message.str.len()
uk["night"] = uk.hh.between(0, 5)
uk["is_weekend"] = uk.date.dt.dayofweek >= 5

authors = uk.groupby("author").agg(
    n=("message", "size"),
    mean_length=("length", "mean"),
    median_length=("length", "median"),
    mean_words=("words", "mean"),
    night_share=("night", "mean"),
    upper_share=("starts_upper", "mean"),
    question_rate=("n_question", "mean"),
    exclaim_rate=("exclaims", "mean"),
    url_share=("has_url", "mean"),
    address_share=("addressed_to", lambda s: s.notna().mean()),
    active_days=("date", "nunique"),
    weekend_share=("is_weekend", "mean"),
)
authors = authors[authors.n >= 30]
authors["msgs_per_day"] = authors.n / authors.active_days
authors["unique_share"] = uk.groupby("author").message.nunique().reindex(authors.index) / authors.n
authors["hour_spread"] = uk.groupby("author").hh.std().reindex(authors.index)

METRICS = [
    "mean_length", "median_length", "mean_words", "night_share", "upper_share",
    "question_rate", "exclaim_rate", "url_share", "address_share", "active_days",
    "msgs_per_day", "unique_share", "hour_spread", "weekend_share", "n",
]

rng = np.random.default_rng(60)
authors["cohort"] = rng.permutation(["A", "B"] * (len(authors) // 2 + 1))[: len(authors)]
print(f"{len(authors)} authors, {len(METRICS)} metrics, cohorts: {authors.cohort.value_counts().to_dict()}")

Test every metric between the cohorts — a Welch t-test each — and put the fifteen p-values on
one axis against the conventional line at 0.05. The one below the line is the finding.

In [ ]:
comparison = []
for metric in METRICS:
    a = authors.loc[authors.cohort == "A", metric].dropna()
    b = authors.loc[authors.cohort == "B", metric].dropna()
    comparison.append({
        "metric": metric,
        "cohort A": a.mean(),
        "cohort B": b.mean(),
        "ratio": a.mean() / b.mean(),
        "p": stats.ttest_ind(a, b, equal_var=False).pvalue,
    })
comparison = pd.DataFrame(comparison).sort_values("p").reset_index(drop=True)
winner = comparison.metric[0]

hunt_plot = PlotSettings(
    figsize=(11, 4),
    title="Fifteen metrics, one of them below the line",
    xlabel="",
    ylabel="p-value (log scale)",
    xtick_rotation=45,
    base_color="#cccccc",
    highlight_color="crimson",
)
bars = BarPlot(hunt_plot)
fig, ax = bars.plot(data=comparison, x="metric", y="p", color=hunt_plot.base_color)
bars.plot_on(HighlightCategory(hunt_plot), categories=[winner])
bars.plot_on(HorizontalLine(hunt_plot), y=0.05, label="p = 0.05", color="black", linewidth=1)
ax.set_yscale("log")
ax.legend()

print(comparison.head(3).round(4).to_string(index=False))

Cohort A does **twice as much of its posting between midnight and 05:00** as cohort B — 13.5%
of their messages against 6.7% — at `p = 0.0026`, on more than two hundred authors per group.

That is a publishable-sounding sentence, and a mechanism suggests itself immediately: cohort
A must be night owls, or in another timezone. Notice how quickly the explanation arrived.

### The reveal

Look at the line that made the cohorts, in the setup cell:

    authors["cohort"] = rng.permutation(["A", "B"] * ...)

There is no cohort. The two groups are the same population, split at random, and every
difference in that plot is produced by nothing at all.

### Why that was going to happen

A p-value of 0.05 means: *if nothing is going on*, a difference this big shows up one time in
twenty. Choose that threshold and you have accepted a 5% false-alarm rate per test. Fifteen
tests, each with a 95% chance of staying quiet, all staying quiet at once:

    P(no false finding in 15 tests) = 0.95^15 = 0.46
    P(at least one)                 = 1 - 0.46 = 0.54

The complement is the trick worth remembering: "at least one" is awkward to count, "none of
them" is a single product. Now measure it instead of deriving it — shuffle the labels three
hundred times and count how many metrics clear 0.05 in each hunt.

In [ ]:
def hunt(seed: int) -> int:
    """Assign cohorts at random, then count the metrics that come out 'significant'."""
    labels = pd.Series(
        np.random.default_rng(seed).permutation(authors.cohort.to_numpy()), index=authors.index
    )
    found = 0
    for metric in METRICS:
        a = authors.loc[labels == "A", metric].dropna()
        b = authors.loc[labels == "B", metric].dropna()
        found += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    return found


findings = pd.Series([hunt(seed) for seed in range(300)], name="findings")
share = findings.value_counts(normalize=True).sort_index().rename("share").reset_index()

null_hunt = PlotSettings(
    figsize=(8, 4),
    title=f"300 hunts over random labels: {(findings > 0).mean():.0%} find something",
    xlabel="metrics below p = 0.05, in one hunt",
    ylabel="share of hunts",
    base_color="#cccccc",
    highlight_color="crimson",
)
bars = BarPlot(null_hunt)
fig, ax = bars.plot(data=share, x="findings", y="share", color=null_hunt.base_color)
_ = bars.plot_on(HighlightCategory(null_hunt), categories=[str(v) for v in share.findings if v > 0])

print(f"measured: a finding in {(findings > 0).mean():.0%} of hunts")
print(f"independence formula: 1 - 0.95^15 = {1 - 0.95 ** 15:.0%}")

Roughly two hunts in five turn up at least one "finding", some turn up two or three, and
all of them are about nothing. **That number is the sanity baseline**, and computing one is the
habit this section teaches: before asking "is my finding real?", ask "how often would I have
found *something* if there were nothing to find?" If the answer is "almost half the time",
finding something is not evidence of anything.

The measured rate is below the 54% the formula predicted, and the reason matters: the formula
assumed fifteen *independent* metrics. They are not — `mean_length`, `median_length` and
`mean_words` are three names for message size; `n`, `active_days` and `msgs_per_day` are three
views of how much someone posts. Fifteen questions are not fifteen chances when several of
them are the same question.

In [ ]:
heat = PlotSettings(figsize=(8, 6.5), title="The fifteen 'independent' metrics", xlabel="", ylabel="")  # ty: ignore[invalid-argument-type]
fig, ax = CorrelationHeatmap(heat).plot(data=authors[METRICS], method="spearman", annot=False)

eigenvalues = np.linalg.eigvalsh(authors[METRICS].corr().fillna(0))
effective = (eigenvalues.sum() ** 2) / (eigenvalues ** 2).sum()
print(f"effective number of independent metrics ≈ {effective:.0f}  →  1 - 0.95^{effective:.0f} = {1 - 0.95 ** effective:.0%}")

Somewhere between nine and eleven real chances, not fifteen. Both directions matter: assuming
independence makes you *over*-estimate how much you hunted, and a baseline that is too high
lets a real finding be dismissed; forgetting the baseline entirely makes you believe the first
striking thing you see.

The honest version of the finding: *"the largest of fifteen differences I looked at, on labels
I did not choose in advance."* That sentence is publishable. "Cohort A are night owls" is not.

### The defence: can you predict the future?

Every fix is a version of the same thing — commit before you look. Write the question down
first, so there is one test. Say how many things you looked at. Or, strongest of all, **split
the data**: hunt on one half, confirm on the other. It is the only fix a reader can check, and
it turns a claim into a prediction: if the pattern is real, it is in data you have not looked
at yet.

Take the night-owl finding to the halves of the corpus the hunt never separated: the same
authors' night share in 2013–2015 and in 2016–2017.

In [ ]:
def night_share(expr: str, feature: str) -> pd.DataFrame:
    """Per-author night share, on only the rows `expr` selects."""
    return (
        Pipeline()
        .add(Filter, expr=expr)
        .add(GroupAgg, by="author", column="night", agg="mean", feature=feature)
        .apply(uk)
    )


held_out = (
    night_share("date.dt.year >= 2013", "whole period")
    .merge(night_share("date.dt.year <= 2015", "2013-2015"), on="author")
    .merge(night_share("date.dt.year >= 2016", "2016-2017"), on="author")
    .join(authors[["cohort"]], on="author")
    .dropna()
)
long = held_out.melt(id_vars=["author", "cohort"], var_name="period", value_name="night share")

for period in ["whole period", "2013-2015", "2016-2017"]:
    a = held_out.loc[held_out.cohort == "A", period]
    b = held_out.loc[held_out.cohort == "B", period]
    print(f"{period:13s} A {a.mean():.3f}  B {b.mean():.3f}   p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.3f}")

predict = PlotSettings(
    figsize=(9, 4),
    title=f"The night-owl gap, on the {len(held_out)} authors active in both halves",
    xlabel="",
    ylabel="share of messages at 00-05h",
)
fig, ax = GroupedBarPlot(predict).plot(
    data=long, x="period", y="night share", hue="cohort",
    palette={"A": "crimson", "B": "#cccccc"}, errorbar=("ci", 95), capsize=0.1,
)
_ = ax.legend(title="cohort", loc="upper left", bbox_to_anchor=(1.01, 1))

Restricted to the authors present in both halves, the gap is already a shadow of the
headline number, and in neither half does it survive: the intervals overlap, the p-values are
nowhere near 0.05. It was never there — random labels cannot predict anything about a period
the hunt did not touch.

> **The rule to carry:** a finding and the search that produced it are one object. Report
> them together, or you have not reported the finding.

## 5.3 What would make you believe it

Run the previous section alone and the conclusion is "nothing is ever real", which is worse
than the credulity it was meant to cure. So: what *would* make a finding believable?

A claim rests on three legs, and statistics supplies exactly one of them.

1. **Evidence** — how many observations, *at the unit the claim is about*, how big is the
   effect, and does it survive a null test.
2. **Mechanism** — is there a reason it would be true, and would you have predicted the
   direction before looking? **No amount of data supplies this leg.**
3. **Replication** — does it appear where you did not look? Another period, another
   channel, a second way of measuring the same thing.

Cross the two cheap ones:

|  | **survives testing** | **fails testing** |
| -- | -- | -- |
| **mechanism** | Report it — then go and check leg three. | **"Plausible, unproven."** Say exactly that, and say what would settle it. |
| **no mechanism** | **Do not conclude. Go looking.** A fluke, or a confounder you have not named — and the confounder is usually the more interesting finding. | Nothing here. Drop it. |

The diagonal is obvious. The off-diagonal cells are the lesson, and both are things students
almost never write down: a null result stated as a null result, and a significant result
treated as a question rather than an answer. Three claims, three cells of the grid, each with
the one plot that decides it.

### Claim 1 · "Release days are busier" — and the baseline is part of the claim

Mechanism: strong. The dates were fixed in advance by an external calendar this dataset had no
say in — ten releases, spread over five years. The obvious test is each release day against a
typical Thursday.

In [ ]:
RELEASES = pd.to_datetime([
    "2013-04-25", "2013-10-17", "2014-04-17", "2014-10-23", "2015-04-23",
    "2015-10-22", "2016-04-21", "2016-10-13", "2017-04-13", "2017-10-19",
])

thursdays = (
    Pipeline()
    .add(Filter, expr="day_name == 'Thursday'")
    .add(FlagDates, column="date", dates=RELEASES, feature="is_release")
    .apply(uk)
)
thursday_counts = Pipeline().add(GroupAgg, by=["date", "is_release"]).apply(thursdays)

global_median = thursday_counts.loc[~thursday_counts.is_release, "n"].median()
release_counts = thursday_counts.loc[thursday_counts.is_release, "n"]
print(f"median ordinary Thursday, whole corpus: {global_median:.0f} messages")
print(f"release days above it: {(release_counts > global_median).sum()}/{len(release_counts)}")

Six out of ten. A coin flip, and on its own it kills the claim.

It should not, and the next claim is the reason: this channel lost most of its traffic across
the window, so "a typical Thursday" in 2013 and in 2017 are different quantities, and the 2017
releases are being compared against a median that mostly comes from 2013. **The trend is a
confounder for the event.** The fix is to compare each release with the Thursdays around it.

In [ ]:
def local_baseline(frame: pd.DataFrame, weeks: int = 4) -> pd.DataFrame:
    """Each release against the median of the Thursdays within `weeks` either side."""
    counts = frame[frame.date.dt.dayofweek == 3].groupby("date").size().sort_index()
    rows = []
    for release_date in RELEASES:
        if release_date not in counts.index:
            continue
        window = counts[(counts.index >= release_date - pd.Timedelta(weeks=weeks))
                        & (counts.index <= release_date + pd.Timedelta(weeks=weeks))]
        rows.append({"release": str(release_date.date()),
                     "nearby Thursday": window.drop(release_date).median(),
                     "on the day": counts[release_date]})
    out = pd.DataFrame(rows)
    out["lift"] = out["on the day"] / out["nearby Thursday"]
    return out


for channel in sorted(msgs.channel.unique()):
    table = local_baseline(msgs[msgs.channel == channel])
    above = int((table.lift > 1).sum())
    p = stats.binomtest(above, len(table), 0.5).pvalue
    print(f"{channel}: {above}/{len(table)} releases above their local baseline, "
          f"median lift {table.lift.median():.2f}x, sign test p = {p:.3f}")

local = local_baseline(uk)
gap = PlotSettings(
    figsize=(9, 5),
    title="#ubuntu-uk: each release against the Thursdays around it",
    xlabel="messages on the day",
    ylabel="",
)
fig, ax = BarbellPlot(gap).plot(
    data=local.sort_values("release", ascending=False), category="release",
    start="nearby Thursday", end="on the day",
    start_label="nearby Thursdays (median)", end_label="release day",
)

**Nine of ten**, median lift 1.7×, `p = 0.02` by a sign test — the right instrument, because
it only asks *which side of typical* each day falls on, which is a question ten noisy days can
answer. Same data, same claim; the only thing that changed was what "typical" means.

**Verdict: report it** — the closest this notebook gets to a finding you could publish.
Mechanism strong, dates fixed by someone else, ten near-independent instances. Leg three is
partial, and say so: `#ubuntu-nl` moves the same way but is a tenth of the size, and a tenth of
the traffic buys a tenth of the resolution. "The direction agrees on a second channel that is
too small to test" is the accurate sentence.

And the detour is the real lesson. The first version said *six out of ten* and would have been
reported as "no effect". A finding can be destroyed by a baseline as easily as invented by one,
which is why §5.2's rule needs its mirror image: **the comparison you chose is part of the
claim.**

### Claim 2 · "The channel got quieter after 2016"

Mechanism: strong, and predictable in advance — Slack, Discord and Matrix happened to IRC
everywhere, not just here. Direction callable before looking. Two channels, each year's volume
relative to its own 2013, on one axis.

In [ ]:
per_year = (
    Pipeline()
    .add(GroupAgg, by=["channel", "year"])
    .apply(msgs.assign(year=msgs.date.dt.year))
)
per_year["relative to 2013"] = per_year.n / per_year.groupby("channel").n.transform("first")

decline = PlotSettings(
    figsize=(9, 4),
    title="Both channels, every year lower than the last",
    xlabel="",
    ylabel="messages, relative to 2013",
)
lines = LinePlot(decline)
fig, ax = lines.plot(data=per_year, x="year", y="relative to 2013", hue="channel", marker="o",
                     palette={"#ubuntu-uk": "steelblue", "#ubuntu-nl": "darkorange"})
lines.plot_on(HorizontalLine(decline), y=1, color="grey", linewidth=0.8)
ax.set_xticks(sorted(per_year.year.unique()))
ax.legend(title="")
print(per_year.pivot(index="year", columns="channel", values="relative to 2013").round(2).to_string())

`#ubuntu-uk` ends at **16%** of its 2013 volume, `#ubuntu-nl` at **8%**, and both fall in
every single year after 2014. Two channels, different countries, different sizes, same
direction, no exceptions.

**Verdict: all three legs.** Mechanism predicted before looking, effect enormous and
monotone, replicated on an independent channel. Nothing here needs a p-value, and asking for
one would be a category error — you are not distinguishing this from noise, you are looking
at it. The honest caveat belongs in the same paragraph: this is *this corpus* getting quieter,
which is not automatically IRC getting quieter, and definitely not the Ubuntu community
shrinking. People moved. The claim is about a channel.

### Claim 3 · "People write longer messages on weekdays"

Mechanism: real, and **ambiguous in sign**. Weekday chat happens at work, in short bursts
between other things — that predicts shorter. Weekday chat is also about work, technical and
detailed — that predicts longer. You genuinely cannot call it in advance, which is when a test
earns its keep.

The trap is the unit. Half a million messages give a p-value with nine zeros; the claim is
about *people*, and 425,000 messages are not 425,000 independent observations of people. Ask it
at the unit the claim is about: one number per author, weekday minus weekend.

In [ ]:
weekday = uk.loc[~uk.is_weekend, "length"]
weekend = uk.loc[uk.is_weekend, "length"]
print(f"per MESSAGE: weekday {weekday.mean():.2f} chars, weekend {weekend.mean():.2f}, "
      f"p = {stats.ttest_ind(weekday, weekend, equal_var=False).pvalue:.0e}  (n = {len(uk):,})")

by_author = uk.groupby(["author", "is_weekend"]).length.mean().unstack().dropna()
by_author = by_author[uk.groupby("author").size().reindex(by_author.index) >= 30]
by_author.columns = pd.Index(["weekday", "weekend"])
by_author["difference"] = by_author.weekday - by_author.weekend

paired = stats.ttest_rel(by_author.weekday, by_author.weekend)
longer = (by_author.difference > 0).mean()
print(f"per AUTHOR:  mean difference {by_author.difference.mean():+.2f} chars, "
      f"paired p = {paired.pvalue:.3f}, longer on weekdays: {longer:.0%}  (n = {len(by_author)})")

split = PlotSettings(
    figsize=(9, 4),
    title=f"One number per author: {longer:.0%} go one way, {1 - longer:.0%} the other",
    xlabel="mean message length, weekday − weekend (characters)",
    ylabel="authors",
)
hist = HistogramPlot(split)
fig, ax = hist.plot(data=by_author.difference.to_numpy(), bins=40, stat="count", color="#cccccc")
_ = hist.plot_on(VerticalLine(split), x=0, color="black", linewidth=1)

259 people, `p = 0.06`, and the histogram is the finding: a coin flip either side of zero.
Whatever is happening is not something individual people do.

**Verdict: plausible, unproven.** Write that sentence down, in a report, as the result:

> *"Message length does not differ meaningfully between weekdays and weekends at the author
> level (n = 259, p = 0.06, and the direction splits 53/47). The per-message difference is
> significant but reflects message counts, not people. Distinguishing them would need more
> authors, or a within-person design across more weeks."*

That is a finished piece of work. Students almost never produce it, because a null reads like
a failed assignment — so they keep slicing until something turns up, which is exactly the hunt
from §5.2. **A null result, honestly bounded, is a pass.**

### Three more, in one row each

The grid has answers for claims this notebook does not need a plot to settle:

| claim | mechanism | evidence | verdict |
| -- | -- | -- | -- |
| "There are far fewer messages at 4am" | overwhelming | 15× | **true, not worth reporting** — nobody's beliefs move; what did you *expect*, and where does the data differ from that? |
| "Nicknames starting A–M write longer messages" | none | nothing, in ten channel-years — and the eight that "agree" are the same authors counted ten times | **drop it** |
| "People on the 4th floor have worse sentiment" (a real submission) | none *for the floor* | every set of numbers has a minimum | **go looking** — the floor is a proxy for a department, a manager, a deadline; the confounder is the finding |

Three claims with a plot and three without, and the answer was different every time. "Be
sceptical" would have got one of them right.

> **What to carry.** Before you write a finding down, say out loud: what is the mechanism,
> what would have surprised me, and where could I check this that I have not already looked?
> If the answer to the first is "none", you are not finished — you are at the beginning of a
> more interesting question. And one question runs under all of it, the same one lesson 4
> answered by shuffling: **what would I expect to see if there were nothing here?**